
# 🧹 The CSV Cleaning Cookbook
### A reusable, general-purpose workflow for importing, auditing, cleaning, and reporting on *any* CSV file

This notebook is built to be dropped into any project. Point `CSV_PATH` at a file, run the
notebook top to bottom, and you get:

| Stage | What happens |
|---|---|
| 📥 **Import** | Robust, auto-detecting `read_csv` wrapper (delimiter, encoding, header, dates) |
| 🔍 **Audit ("Before")** | Shape, dtypes, memory, missingness, duplicates, cardinality, outliers |
| 🧽 **Clean** | Configurable pipeline: whitespace, dtype coercion, missing-value handling, dedup, outlier flags |
| 📊 **Audit ("After")** | Same audit re-run on the cleaned data |
| ⚖️ **Before vs. After** | Side-by-side tables + charts showing exactly what changed |
| 💾 **Export** | Cleaned CSV + a saved text/HTML data-quality report |

> **How to use it:** edit the **⚙️ Configuration** cell only — the rest of the notebook is
> written to be file-agnostic and just works. Every function is documented and safe to
> re-run (idempotent where possible).

---



## 0. Setup

Install/import everything we need. `matplotlib` + `seaborn` are used only for the
before/after visuals — the notebook still works without them (charts are skipped
gracefully if missing).


In [ ]:

import sys, subprocess

# Make sure optional plotting libs are present (no-op if already installed)
for pkg in ("seaborn",):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    plt.rcParams["figure.facecolor"] = "white"
    sns.set_style("whitegrid")
except ImportError:
    HAS_PLOTS = False
    print("matplotlib/seaborn not available — charts will be skipped, tables still work.")

# ---- Display options (readable floats, wide enough columns) ----
pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

print("Environment ready ✅   |   pandas", pd.__version__)



## ⚙️ 1. Configuration — *the only cell you should need to edit*

Point this at any CSV. Every setting has a sensible default and `read_csv_robust`
(next section) will auto-detect what it can — you only need to override what you know
in advance (e.g. custom column names, or specific date columns).


In [ ]:

CONFIG = {
    # ---- Required ----
    "CSV_PATH": "data/landtempssample.csv",   # <-- change this

    # ---- Optional import overrides (None = auto-detect) ----
    "SEP": None,                 # e.g. ',' ';' '\t'  -> None = auto-detect
    "ENCODING": None,             # e.g. 'utf-8', 'latin1' -> None = auto-detect
    "COLUMN_NAMES": None,         # e.g. ['stationid','year','month',...] -> None = use file header
    "SKIPROWS": None,             # e.g. 1 if COLUMN_NAMES replaces a header row
    "DATE_COLUMNS": None,         # e.g. [['month','year']] or ['order_date'] -> None = auto-detect
    "DTYPE_OVERRIDES": None,      # e.g. {'stationid': 'string'}

    # ---- Cleaning behavior ----
    "TRIM_WHITESPACE": True,
    "STANDARDIZE_COLUMN_NAMES": True,   # lower_snake_case
    "DROP_FULLY_EMPTY_ROWS": True,
    "DROP_COLS_MISSING_ABOVE": 0.90,    # drop a column if >90% missing (None to disable)
    "DEDUPLICATE_ROWS": True,
    "DEDUP_SUBSET": None,               # None = consider all columns
    "CRITICAL_COLUMNS": [],             # rows missing these are dropped, e.g. ['avgtemp']
    "FLAG_OUTLIERS_IQR": True,          # adds is_outlier_<col> boolean columns (non-destructive)

    # ---- Output ----
    "OUTPUT_DIR": "/mnt/user-data/outputs",
}

print("Config loaded. Editing CONFIG['CSV_PATH'] is usually all you need.")



## 2. 📥 Robust CSV Import

`read_csv` almost always needs *some* help. `read_csv_robust()` wraps it and
auto-detects the pieces people usually get wrong on the first try:

- **Delimiter** — sniffed from the first few KB with `csv.Sniffer`
- **Encoding** — tries `utf-8`, falls back to `latin1`, then `cp1252`
- **Date-like columns** — any object column where >70% of a sample parses as a date
- **Low-memory chunking** — always off (`low_memory=False`) to avoid the classic
  *"mixed dtype in column"* warning on large files

This mirrors the pattern from *Importing CSV files* (this chapter): explicit
`names`, `skiprows`, and `parse_dates` when you know them, sensible auto-detection
when you don't.


In [ ]:

import csv as _csv
import io

def _sniff_separator(path: str, encoding: str) -> str:
    """Guess the delimiter by sampling the first chunk of the file."""
    with open(path, "r", encoding=encoding, errors="replace") as f:
        sample = f.read(65536)
    try:
        dialect = _csv.Sniffer().sniff(sample, delimiters=",;\t|")
        return dialect.delimiter
    except _csv.Error:
        return ","  # sensible fallback


def _sniff_encoding(path: str) -> str:
    """Try common encodings in order of likelihood; return the first that reads cleanly."""
    for enc in ("utf-8", "utf-8-sig", "latin1", "cp1252"):
        try:
            with open(path, "r", encoding=enc) as f:
                f.read(65536)
            return enc
        except (UnicodeDecodeError, LookupError):
            continue
    return "utf-8"  # last resort; pandas will replace bad bytes downstream


def _guess_date_columns(df: pd.DataFrame, sample_size: int = 200, hit_rate: float = 0.7) -> list:
    """Flag object columns that look like dates without forcing a parse yet."""
    candidates = []
    for col in df.select_dtypes(include="object").columns:
        sample = df[col].dropna().astype(str).head(sample_size)
        if sample.empty:
            continue
        parsed = pd.to_datetime(sample, errors="coerce", format="mixed")
        if parsed.notna().mean() >= hit_rate:
            candidates.append(col)
    return candidates


def read_csv_robust(cfg: dict) -> pd.DataFrame:
    """
    General-purpose CSV loader driven by a CONFIG dict.
    Auto-detects separator / encoding / date columns unless overridden.
    """
    path = cfg["CSV_PATH"]
    encoding = cfg.get("ENCODING") or _sniff_encoding(path)
    sep = cfg.get("SEP") or _sniff_separator(path, encoding)

    read_kwargs = dict(
        filepath_or_buffer=path,
        sep=sep,
        encoding=encoding,
        low_memory=False,
        encoding_errors="replace",
    )
    if cfg.get("COLUMN_NAMES"):
        read_kwargs["names"] = cfg["COLUMN_NAMES"]
        read_kwargs["skiprows"] = cfg.get("SKIPROWS", 1)
    if cfg.get("DTYPE_OVERRIDES"):
        read_kwargs["dtype"] = cfg["DTYPE_OVERRIDES"]

    # First pass: load without forcing date parsing so we can inspect + sniff dates.
    df = pd.read_csv(**read_kwargs)

    date_cols = cfg.get("DATE_COLUMNS")
    if date_cols is None:
        date_cols = _guess_date_columns(df)

    if date_cols:
        for dc in date_cols:
            if isinstance(dc, list):
                # combined columns, e.g. ['month','year'] -> parse then merge
                combo_name = "_".join(dc)
                try:
                    df[combo_name] = pd.to_datetime(
                        df[dc].astype(str).agg("-".join, axis=1), errors="coerce", format="mixed"
                    )
                    df.drop(columns=dc, inplace=True)
                except Exception:
                    pass
            elif dc in df.columns:
                df[dc] = pd.to_datetime(df[dc], errors="coerce", format="mixed")

    meta = {"sep": sep, "encoding": encoding, "date_columns_detected": date_cols}
    return df, meta


raw_df, load_meta = read_csv_robust(CONFIG)

print(f"Loaded: {CONFIG['CSV_PATH']}")
print(f"Detected separator: {load_meta['sep']!r}   |   encoding: {load_meta['encoding']}")
print(f"Detected/used date columns: {load_meta['date_columns_detected']}")
print(f"Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
raw_df.head(7)



## 3. 🔍 Reusable Data-Quality Audit Toolkit

These functions work on *any* DataFrame — we'll call the same `profile()` function
on the raw data (**Before**) and the cleaned data (**After**) so the comparison in
Section 6 is apples-to-apples.


In [ ]:

def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    counts = df.isnull().sum()
    pct = (counts / len(df) * 100).round(2)
    out = pd.DataFrame({"missing_count": counts, "missing_pct": pct})
    return out.sort_values("missing_pct", ascending=False)


def dtype_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.dropna().size else None),
    })


def duplicate_summary(df: pd.DataFrame) -> dict:
    return {
        "full_row_duplicates": int(df.duplicated().sum()),
        "pct_of_rows": round(df.duplicated().mean() * 100, 2),
    }


def numeric_outlier_summary(df: pd.DataFrame) -> pd.DataFrame:
    """IQR-method outlier counts for every numeric column (report only, non-destructive)."""
    rows = []
    for col in df.select_dtypes(include="number").columns:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((df[col] < lo) | (df[col] > hi)).sum()
        rows.append({"column": col, "lower_bound": lo, "upper_bound": hi,
                      "n_outliers": int(n_out), "pct_outliers": round(n_out / len(df) * 100, 2)})
    return pd.DataFrame(rows).set_index("column") if rows else pd.DataFrame()


def profile(df: pd.DataFrame, label: str = "") -> dict:
    """Bundle every audit into one dict so it can be snapshotted (Before) and (After)."""
    return {
        "label": label,
        "shape": df.shape,
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 3),
        "dtypes": dtype_summary(df),
        "missing": missing_summary(df),
        "duplicates": duplicate_summary(df),
        "outliers": numeric_outlier_summary(df),
        "describe_numeric": df.describe().T,
        "describe_categorical": df.select_dtypes(include="object").describe().T
                                  if not df.select_dtypes(include="object").empty else pd.DataFrame(),
    }


def print_profile(p: dict):
    print("=" * 72)
    print(f" DATA PROFILE: {p['label']}")
    print("=" * 72)
    print(f"Shape        : {p['shape'][0]:,} rows x {p['shape'][1]} columns")
    print(f"Memory       : {p['memory_mb']:,} MB")
    print(f"Duplicates   : {p['duplicates']['full_row_duplicates']:,} rows "
          f"({p['duplicates']['pct_of_rows']}%)")
    print("\n--- dtypes ---")
    print(p["dtypes"])
    print("\n--- missing values (columns with >0 missing) ---")
    miss = p["missing"]
    print(miss[miss.missing_count > 0] if not miss[miss.missing_count > 0].empty else "  none")
    if not p["outliers"].empty:
        print("\n--- numeric outliers (IQR method) ---")
        print(p["outliers"])

print("Audit toolkit ready: profile(), print_profile(), missing_summary(), "
      "dtype_summary(), duplicate_summary(), numeric_outlier_summary()")



### 3.1 Run the **"Before"** audit


In [ ]:

before_profile = profile(raw_df, label="BEFORE cleaning")
print_profile(before_profile)



### 3.2 Visual first look — missingness & distributions (Before)

A quick visual scan often surfaces problems a text summary hides (e.g. a column that's
*mostly* fine but has one wild outlier that wrecks a chart's scale).


In [ ]:

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Missingness heatmap
    sns.heatmap(raw_df.isnull(), cbar=False, yticklabels=False,
                cmap=sns.color_palette(["#e8eaf0", "#e4572e"]), ax=axes[0])
    axes[0].set_title("Missing values map (orange = missing) — BEFORE")

    # Missing % bar chart
    miss_pct = before_profile["missing"]["missing_pct"]
    miss_pct = miss_pct[miss_pct > 0].sort_values()
    if not miss_pct.empty:
        axes[1].barh(miss_pct.index, miss_pct.values, color="#e4572e")
        axes[1].set_xlabel("% missing")
        axes[1].set_title("Missing % by column — BEFORE")
    else:
        axes[1].text(0.5, 0.5, "No missing values 🎉", ha="center", va="center")
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib/seaborn to see the visual audit.")



## 4. 🧽 Cleaning Pipeline

Each step below is its own small, well-named function so you can reorder, skip, or
reuse individual steps in other projects. `run_cleaning_pipeline()` chains them
according to the flags set in `CONFIG`.


In [ ]:

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """snake_case, no leading/trailing whitespace, no duplicate separators."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(r"[^\w]+", "_", regex=True)
                  .str.strip("_")
    )
    return df


def trim_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    """Strip leading/trailing whitespace from every string cell."""
    df = df.copy()
    obj_cols = df.select_dtypes(include="object").columns
    for col in obj_cols:
        df[col] = df[col].str.strip()
    return df


def drop_empty_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(how="all")


def drop_sparse_columns(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Drop columns whose missing fraction exceeds `threshold` (e.g. 0.9 = 90%)."""
    if threshold is None:
        return df
    keep_thresh = int(len(df) * (1 - threshold))
    return df.dropna(axis=1, thresh=keep_thresh)


def drop_critical_missing(df: pd.DataFrame, critical_cols: list) -> pd.DataFrame:
    cols = [c for c in critical_cols if c in df.columns]
    return df.dropna(subset=cols) if cols else df


def deduplicate(df: pd.DataFrame, subset=None) -> pd.DataFrame:
    return df.drop_duplicates(subset=subset)


def flag_outliers_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """Non-destructive: adds is_outlier_<col> boolean flag columns for numeric fields."""
    df = df.copy()
    for col in df.select_dtypes(include="number").columns:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"is_outlier_{col}"] = ~df[col].between(lo, hi)
    return df


def run_cleaning_pipeline(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Apply the configured cleaning steps in a sensible, safe order."""
    steps_log = []
    out = df.copy()

    if cfg.get("STANDARDIZE_COLUMN_NAMES"):
        out = standardize_column_names(out)
        steps_log.append("standardized column names")

    if cfg.get("TRIM_WHITESPACE"):
        out = trim_whitespace(out)
        steps_log.append("trimmed whitespace on text columns")

    if cfg.get("DROP_FULLY_EMPTY_ROWS"):
        before_n = len(out)
        out = drop_empty_rows(out)
        steps_log.append(f"dropped {before_n - len(out):,} fully-empty rows")

    if cfg.get("DROP_COLS_MISSING_ABOVE") is not None:
        before_c = out.shape[1]
        out = drop_sparse_columns(out, cfg["DROP_COLS_MISSING_ABOVE"])
        steps_log.append(f"dropped {before_c - out.shape[1]} columns "
                          f">{cfg['DROP_COLS_MISSING_ABOVE']*100:.0f}% missing")

    if cfg.get("CRITICAL_COLUMNS"):
        before_n = len(out)
        out = drop_critical_missing(out, cfg["CRITICAL_COLUMNS"])
        steps_log.append(f"dropped {before_n - len(out):,} rows missing critical columns "
                          f"{cfg['CRITICAL_COLUMNS']}")

    if cfg.get("DEDUPLICATE_ROWS"):
        before_n = len(out)
        out = deduplicate(out, cfg.get("DEDUP_SUBSET"))
        steps_log.append(f"removed {before_n - len(out):,} duplicate rows")

    if cfg.get("FLAG_OUTLIERS_IQR"):
        out = flag_outliers_iqr(out)
        steps_log.append("added is_outlier_<col> flags (IQR method, non-destructive)")

    for s in steps_log:
        print(" •", s)

    return out


print("Cleaning pipeline ready. Running it now...\n")
clean_df = run_cleaning_pipeline(raw_df, CONFIG)
print(f"\nFinal shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns")
clean_df.head(7)



## 5. 🔍 Re-run the audit on the cleaned data ("After")


In [ ]:

after_profile = profile(clean_df, label="AFTER cleaning")
print_profile(after_profile)



## 6. ⚖️ Before vs. After — Side-by-Side Comparison

This is the payoff: a single table and a couple of charts that make it obvious what
the cleaning pipeline actually did.


In [ ]:

def build_comparison_table(before: dict, after: dict) -> pd.DataFrame:
    rows = [
        ("Rows", before["shape"][0], after["shape"][0]),
        ("Columns", before["shape"][1], after["shape"][1]),
        ("Memory (MB)", before["memory_mb"], after["memory_mb"]),
        ("Duplicate rows", before["duplicates"]["full_row_duplicates"],
                            after["duplicates"]["full_row_duplicates"]),
        ("Columns with missing values",
             int((before["missing"]["missing_count"] > 0).sum()),
             int((after["missing"]["missing_count"] > 0).sum())),
        ("Total missing cells",
             int(before["missing"]["missing_count"].sum()),
             int(after["missing"]["missing_count"].sum())),
    ]
    comp = pd.DataFrame(rows, columns=["Metric", "Before", "After"])
    comp["Change"] = comp["After"] - comp["Before"]
    return comp


comparison = build_comparison_table(before_profile, after_profile)

# Nicely styled table for notebook display
(comparison.style
    .format({"Before": "{:,.2f}", "After": "{:,.2f}", "Change": "{:+,.2f}"})
    .background_gradient(subset=["Change"], cmap="RdYlGn_r")
    .set_caption("📋 Before vs. After — Key Metrics")
    .set_table_styles([{"selector": "caption",
                         "props": [("font-size", "14px"), ("font-weight", "bold")]}])
)


In [ ]:

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # 1) Row count before/after
    axes[0].bar(["Before", "After"], [before_profile["shape"][0], after_profile["shape"][0]],
                color=["#94a3b8", "#4c956c"])
    axes[0].set_title("Row count")
    for i, v in enumerate([before_profile["shape"][0], after_profile["shape"][0]]):
        axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

    # 2) Missing cells before/after
    miss_before = int(before_profile["missing"]["missing_count"].sum())
    miss_after = int(after_profile["missing"]["missing_count"].sum())
    axes[1].bar(["Before", "After"], [miss_before, miss_after],
                color=["#e4572e", "#4c956c"])
    axes[1].set_title("Total missing cells")
    for i, v in enumerate([miss_before, miss_after]):
        axes[1].text(i, v, f"{v:,}", ha="center", va="bottom")

    # 3) Memory footprint before/after
    axes[2].bar(["Before", "After"], [before_profile["memory_mb"], after_profile["memory_mb"]],
                color=["#94a3b8", "#4c956c"])
    axes[2].set_title("Memory (MB)")
    for i, v in enumerate([before_profile["memory_mb"], after_profile["memory_mb"]]):
        axes[2].text(i, v, f"{v:,.2f}", ha="center", va="bottom")

    plt.suptitle("Before vs. After Cleaning", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib/seaborn to see the comparison charts.")



### 6.1 Missingness — column by column, Before vs. After


In [ ]:

miss_compare = (
    before_profile["missing"][["missing_pct"]].rename(columns={"missing_pct": "before_pct"})
    .join(after_profile["missing"][["missing_pct"]].rename(columns={"missing_pct": "after_pct"}),
          how="outer")
    .fillna(0)
    .sort_values("before_pct", ascending=False)
)

if HAS_PLOTS and not miss_compare.empty:
    ax = miss_compare.plot(kind="barh", figsize=(9, max(3, 0.35 * len(miss_compare))),
                            color=["#e4572e", "#4c956c"])
    ax.set_xlabel("% missing")
    ax.set_title("Missing % by column — Before vs. After")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

miss_compare



### 6.2 Distribution sanity check — did cleaning distort the numbers?

Overlaying Before/After histograms for numeric columns is the fastest way to confirm
that dropping rows/columns didn't silently bias the dataset.


In [ ]:

if HAS_PLOTS:
    numeric_cols = [c for c in raw_df.select_dtypes(include="number").columns
                     if c in clean_df.columns][:6]  # cap at 6 for readability

    if numeric_cols:
        n = len(numeric_cols)
        cols = 3
        rows = -(-n // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
        axes = axes.flatten() if n > 1 else [axes]

        for i, col in enumerate(numeric_cols):
            axes[i].hist(raw_df[col].dropna(), bins=30, alpha=0.5, label="Before", color="#94a3b8")
            axes[i].hist(clean_df[col].dropna(), bins=30, alpha=0.6, label="After", color="#4c956c")
            axes[i].set_title(col)
            axes[i].legend(fontsize=8)

        for j in range(len(numeric_cols), len(axes)):
            axes[j].axis("off")

        plt.suptitle("Distribution shape — Before vs. After", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("No shared numeric columns to compare.")
else:
    print("Install matplotlib/seaborn to see distribution comparisons.")



## 7. 💾 Export cleaned data + a saved data-quality report

Writes:
- `<name>_cleaned.csv` — the cleaned DataFrame
- `<name>_data_quality_report.txt` — a plain-text audit report (Before, After, and the diff),
  suitable for pasting into a PR description or data-handoff email.


In [ ]:

def write_report(before: dict, after: dict, comparison: pd.DataFrame, cfg: dict) -> str:
    out_dir = Path(cfg["OUTPUT_DIR"])
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(cfg["CSV_PATH"]).stem

    lines = []
    lines.append(f"DATA QUALITY REPORT — {stem}")
    lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"Source file: {cfg['CSV_PATH']}")
    lines.append("=" * 72)
    lines.append("\nSUMMARY OF CHANGES\n" + "-" * 72)
    lines.append(comparison.to_string(index=False))

    lines.append("\n\nBEFORE — missing values by column\n" + "-" * 72)
    miss_b = before["missing"]
    lines.append(miss_b[miss_b.missing_count > 0].to_string()
                  if not miss_b[miss_b.missing_count > 0].empty else "  none")

    lines.append("\n\nAFTER — missing values by column\n" + "-" * 72)
    miss_a = after["missing"]
    lines.append(miss_a[miss_a.missing_count > 0].to_string()
                  if not miss_a[miss_a.missing_count > 0].empty else "  none")

    if not after["outliers"].empty:
        lines.append("\n\nAFTER — numeric outliers (IQR method, flagged not removed)\n" + "-" * 72)
        lines.append(after["outliers"].to_string())

    report_text = "\n".join(lines)
    report_path = out_dir / f"{stem}_data_quality_report.txt"
    report_path.write_text(report_text)

    csv_path = out_dir / f"{stem}_cleaned.csv"
    clean_df.to_csv(csv_path, index=False)

    print(f"Cleaned data saved to : {csv_path}")
    print(f"Report saved to       : {report_path}")
    return str(csv_path), str(report_path)


cleaned_csv_path, report_path = write_report(before_profile, after_profile, comparison, CONFIG)



## 8. 📚 Quick-Reference Cheat Sheet

A summary of the reusable pieces in this notebook, and the classic `read_csv` gotchas
they solve (see also *"Anticipating Data Cleaning Issues when Importing Tabular Data
into pandas"*).

| Function | Purpose | Solves |
|---|---|---|
| `read_csv_robust(cfg)` | Load any CSV with auto-detected separator/encoding/dates | Wrong delimiter, mixed encodings, dates left as text |
| `profile(df, label)` | One-call snapshot: shape, dtypes, missing, duplicates, outliers | "What does this dataset actually look like?" |
| `standardize_column_names(df)` | `lower_snake_case` headers | Inconsistent / messy column names |
| `trim_whitespace(df)` | Strips stray spaces from text cells | `"US "` != `"US"` bugs in joins/filters |
| `drop_empty_rows(df)` | Removes fully-blank rows | Trailing blank lines in exports |
| `drop_sparse_columns(df, thresh)` | Drops columns that are mostly empty | Columns not worth keeping |
| `drop_critical_missing(df, cols)` | Drops rows missing must-have fields | e.g. no `avgtemp` value at all |
| `deduplicate(df, subset)` | Removes duplicate rows | Re-exported / re-scraped data |
| `flag_outliers_iqr(df)` | Adds `is_outlier_<col>` flags (non-destructive) | Extreme values that skew stats/plots |
| `build_comparison_table(before, after)` | Before/after metrics in one table | Communicating what cleaning changed |
| `write_report(...)` | Saves a plain-text audit trail | Reproducibility / handoff documentation |

### Common `read_csv` options worth remembering
```python
pd.read_csv(
    path,
    sep=";",                       # non-comma delimiter
    names=[...], skiprows=1,       # replace a messy header row
    parse_dates=[["month","year"]],# combine + parse date columns
    dtype={"id": "string"},        # force a dtype
    low_memory=False,              # avoid mixed-dtype warnings on big files
    encoding="latin1",             # non-UTF8 source files
    compression="zip",             # read directly from a .zip
)
```

### Re-using this notebook
1. Change `CONFIG["CSV_PATH"]` (and any overrides you already know) in **Section 1**.
2. *Restart & Run All.*
3. Read the **Before vs. After** table/charts in **Section 6**.
4. Grab the cleaned CSV + text report from `CONFIG["OUTPUT_DIR"]`.
